In [15]:
import tiktoken
import json
from pathlib import Path




In [16]:
# def estimate_request_tokens(record: dict, encoder: tiktoken.Encoding) -> int:
#     body = record.get("body", {})
#     messages = body.get("messages", [])
#     text_parts: list[str] = []
#     print(body)
#     print(messages)

#     for msg in messages:
#         content = msg.get("content", "")
#         if isinstance(content, list):
#             for part in content:
#                 if isinstance(part, str):
#                     text_parts.append(part)
#                 elif isinstance(part, dict) and "text" in part:
#                     text_parts.append(str(part["text"]))
#         else:
#             text_parts.append(str(content))

#     input_tokens = len(encoder.encode("".join(text_parts)))
#     max_tokens = int(body.get("max_tokens", 0) or 0)
#     return input_tokens + max_tokens

def estimate_request_tokens(record: dict, encoder: tiktoken.Encoding) -> int:
    body = record.get("body", {})
    messages = body.get("messages", [])
    
    input_tokens = len(encoder.encode("".join(str(msg.get("content", "")) for msg in messages)))
    max_tokens = int(body.get("max_tokens", 0) or 0)
    
    return input_tokens + max_tokens

def save_chunk(output_dir: Path, index: int, lines: list[str]) -> Path:
    pass

def split_batch_file(
    batch_file_path: Path,
    output_dir: Path,
    max_batch_tokens: int = 85000,
) -> list[Path]:
    if not batch_file_path.exists():
        raise FileNotFoundError(f"{batch_file_path} does not exist.")

    # output_dir.mkdir(parents=True, exist_ok=True)
    encoder = tiktoken.get_encoding("o200k_base")

    chunks: list[Path] = []
    current_chunk: list[str] = []
    current_tokens = 0
    chunk_index = 1

    with batch_file_path.open("r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            print(record)
            print(encoder)
            
            cost = estimate_request_tokens(record, encoder)
            print(cost)
            return

            if cost > max_batch_tokens:
                raise ValueError(
                    f"Single request {record.get('custom_id')} needs {cost} tokens, "
                    f"exceeds chunk limit {max_batch_tokens}"
                )

            if current_tokens + cost > max_batch_tokens and current_chunk:
                chunk_path = save_chunk(output_dir, chunk_index, current_chunk)
                chunks.append(chunk_path)
                chunk_index += 1
                current_chunk = []
                current_tokens = 0

            current_chunk.append(line)
            current_tokens += cost

    if current_chunk:
        chunk_path = save_chunk(output_dir, chunk_index, current_chunk)
        chunks.append(chunk_path)

    return chunks

In [17]:
split_batch_file(Path("../data/train/teacher_batch_input.jsonl"), Path("../data/train/batches_input.jsonl"))

{'custom_id': 'fb-trace-financebench_id_01487', 'method': 'POST', 'url': '/v1/chat/completions', 'body': {'model': 'gpt-4o', 'temperature': 0.3, 'max_tokens': 1000, 'messages': [{'role': 'system', 'content': 'You are an expert financial analyst.'}, {'role': 'user', 'content': '\n                You are given a question about a company\'s financial statements, the relevant evidence from the official filing, and the **correct** answer. \n\n                Your job is to explain, step-by-step, how to derive that answer using ONLY the provided evidence.\n\n\n\n                Question: Did JnJ\'s net earnings as a percent of sales increase in Q2 of FY2023 compared to Q2 of FY2022?\n\n                Evidence: [{\'evidence_text\': \'Johnson & Johnson and Subsidiaries\\n \\nCondensed Consolidated Statement of Earnings \\n \\n(Unaudited; in Millions Except Per Share Figures)\\nPercent\\nPercent\\nPercent\\nIncrease\\nAmount\\nto Sales\\nAmount\\nto Sales\\n(Decrease)\\nSales to customers\\n25